# Stage 4 — Readability & Faithfulness Analysis

**Reads:** `outputs/explanations.json`  
**Writes:** `outputs/results.csv`

Computes per-explanation metrics:

| Metric | Description |
|--------|-------------|
| `flesch_reading_ease` | Higher = simpler text |
| `flesch_kincaid_grade` | Higher = more complex |
| `smog_index` | Estimated years of education needed |
| `lime_coverage` | Fraction of LIME features referenced in explanation |
| `ontology_hit_rate` | Fraction of features mapped to ≥1 ancestor |

> **Tip:** This notebook is fast — safe to re-run freely as you tweak display cells.

## 1. Imports

In [1]:
import pandas as pd

from config import ANALYSIS_RESULTS_PATH, EXPLANATIONS_PATH
from pipeline_helpers import (
    checkpoint_exists,
    lime_coverage,
    load_checkpoint,
    ontology_hit_rate,
    readability_metrics,
    save_checkpoint,
)

## 2. Configuration

In [2]:
# Set True to recompute even if results.csv already exists
FORCE_RERUN = True

## 3. Load Stage 3 output

In [3]:
if not EXPLANATIONS_PATH.exists():
    raise FileNotFoundError(
        f"Explanations not found at '{EXPLANATIONS_PATH}'.\n"
        "Please run 03_llm.ipynb first."
    )
data = load_checkpoint(EXPLANATIONS_PATH)
print(f"Loaded {len(data)} explanations.")

[Checkpoint] Loaded 50 records ← 'outputs\ex_exp_cd.json'
Loaded 50 explanations.


## 4. Compute metrics

In [4]:
rows = []
for item in data:
    explanation  = item.get("explanation", "")
    feature_data = item.get("feature_data", [])

    row = {
        "text":              item["text"][:80] + "…",
        "predicted_class":   item["predicted_class"],
        "confidence":        item["confidence"],
        "user_category":     item["user_category"],
        "ablation_mode":     item["ablation_mode"],
        "explanation":       explanation,
        "lime_coverage":     lime_coverage(explanation, feature_data),
        "ontology_hit_rate": ontology_hit_rate(feature_data),
        **readability_metrics(explanation),
    }
    rows.append(row)

df = pd.DataFrame(rows)
print(f"✅ Metrics computed for {len(df)} explanations.")

✅ Metrics computed for 50 explanations.


## 5. Full results table

In [5]:
pd.set_option("display.max_colwidth", 60)
df[[
    "predicted_class", "user_category", "ablation_mode",
    "flesch_reading_ease", "flesch_kincaid_grade", "smog_index",
    "lime_coverage", "ontology_hit_rate"
]]

,predicted_class,user_category,ablation_mode,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
0,Digestive system diseases,EXPERT,normal,8.708344,18.106328,18.026120,0.0000,1.0
1,Cardiovascular diseases,EXPERT,normal,38.198696,13.645217,14.554593,0.0000,1.0
2,Digestive system diseases,EXPERT,normal,12.318889,18.248642,19.287187,0.3333,1.0
3,Cardiovascular diseases,EXPERT,normal,8.458794,18.588305,18.878055,1.0000,1.0
4,Neoplasms,EXPERT,normal,17.382143,14.312857,14.554593,0.0000,0.0
5,Neoplasms,EXPERT,normal,41.349598,12.294828,13.816670,1.0000,1.0
6,General pathological conditions,EXPERT,normal,-10.330000,18.675000,18.243606,0.0000,0.0
7,Neoplasms,EXPERT,normal,32.163333,15.480741,15.903189,1.0000,1.0
8,General pathological conditions,EXPERT,normal,23.850000,15.895000,17.505863,1.0000,1.0
9,Digestive system diseases,EXPERT,normal,5.532500,16.462500,17.122413,0.0000,0.0


## 6. Summary — mean metrics by user category & ablation mode

In [6]:
metric_cols = [
    "flesch_reading_ease", "flesch_kincaid_grade",
    "smog_index", "lime_coverage", "ontology_hit_rate",
]
summary = (
    df.groupby(["user_category", "ablation_mode"])[metric_cols]
    .mean()
    .round(3)
)
summary

,,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
user_category,ablation_mode,,,,,
EXPERT,normal,14.349,16.469,16.799,0.57,0.64


## 7. [Optional] Per-class breakdown

In [7]:
df.groupby("predicted_class")[metric_cols].mean().round(3)

,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
predicted_class,,,,,
Cardiovascular diseases,9.145,17.268,16.964,0.600,0.667
Digestive system diseases,15.876,16.801,17.486,0.667,0.818
General pathological conditions,0.881,17.962,18.046,0.396,0.500
Neoplasms,26.458,14.626,15.554,0.615,0.615
Nervous system diseases,18.208,15.260,15.532,0.333,0.333


## 8. [Optional] Read a specific explanation in full

In [8]:
# Change the index to read any explanation in full
IDX = 0

row = df.iloc[IDX]
print(f"Text     : {row['text']}")
print(f"Class    : {row['predicted_class']} ({row['confidence']})")
print(f"User     : {row['user_category']}")
print(f"Ablation : {row['ablation_mode']}")
print(f"\n── Explanation ────────────────────────────────────")
print(row["explanation"])

Text     : Normalization of ventilation/perfusion relationships after liver transplantation…
Class    : Digestive system diseases (0.5919)
User     : EXPERT
Ablation : normal

── Explanation ────────────────────────────────────
The model classified the biomedical abstract as "Digestive System Diseases" due to the presence of key words and phrases related to the cardiovascular and respiratory systems. The abstract mentions "cardiac output" and "pulmonary vascular pressures," which are directly associated with the cardiovascular system. Additionally, the mention of "arteries" is crucial because arteries are part of the circulatory system, which plays a vital role in delivering oxygen and nutrients to the body's tissues and removing waste products. The cardiovascular system is closely linked to the digestive system, as both systems work together to maintain overall bodily function. Therefore, the abstract's content is strongly indicative of diseases affecting the cardiovascular or respira

## 9. Save to CSV

In [9]:
df.to_csv(ANALYSIS_RESULTS_PATH, index=False)
print(f"✅ Saved {len(df)} rows → '{ANALYSIS_RESULTS_PATH}'")

✅ Saved 50 rows → 'outputs\res_exp_cd.csv'
